Аналіз датасету споживання електроенергії
Завантаження датасету, очищення даних (видалення пропусків) та об'єднання дати і часу.

In [1]:
import pandas as pd
import urllib.request
import zipfile
import os

# Посилання на офіційний архів датасету
url = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
zip_path = "power_consumption.zip"
txt_path = "household_power_consumption.txt"

# Перевіряємо, чи є вже файл
if not os.path.exists(txt_path):
    print("Завантажуємо архів (близько 20 МБ)...")
    urllib.request.urlretrieve(url, zip_path)
    print("Розпаковуємо архів...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()
    print("Розпакування завершено!")

print("Читаємо дані у DataFrame (це може зайняти хвилину, таблиця величезна)...")
# Файл розділений крапкою з комою. Пропуски в ньому записані як '?'
df = pd.read_csv(txt_path, sep=';', na_values='?', low_memory=False)

# Data Cleaning
print("Очищаємо дані...")
df = df.dropna() 

# Об'єднуємо Date та Time в одну колонку
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')

print("Готово!")
display(df.head())

Завантажуємо архів (близько 20 МБ)...
Розпаковуємо архів...
Розпакування завершено!
Читаємо дані у DataFrame (це може зайняти хвилину, таблиця величезна)...
Очищаємо дані...
Готово!


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт


In [2]:
def filter_high_power(df):
    # Фільтруємо таблицю 
    return df[df['Global_active_power'] > 5.0]

print("Записи з потужністю > 5 кВт:")
high_power_df = filter_high_power(df)
display(high_power_df.head())

print("\nЧас виконання процедури:")
%timeit filter_high_power(df)

Записи з потужністю > 5 кВт:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0,2006-12-16 17:35:00
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0,2006-12-16 17:36:00



Час виконання процедури:
4.97 ms ± 171 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер

In [3]:
def filter_appliances(df):
    cond_intensity = (df['Global_intensity'] >= 19.0) & (df['Global_intensity'] <= 20.0)
    cond_appliances = df['Sub_metering_2'] > df['Sub_metering_3']

    return df[cond_intensity & cond_appliances]

print("Результат вибірки:")
filtered_df = filter_appliances(df)
display(filtered_df.head())

print("\nЧас виконання процедури:")
%timeit filter_appliances(df)

Результат вибірки:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0,2006-12-17 01:20:00



Час виконання процедури:
9.38 ms ± 895 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії


In [4]:
def random_sample_means(df):
    # 500 000 випадкових рядків без повторень
    sample_df = df.sample(n=500000, replace=False, random_state=42)

    # Середнє 
    means = sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return means

print("Середні величини для 500 000 випадкових записів:")
# Обертаємо в DataFrame
display(pd.DataFrame(random_sample_means(df), columns=['Середнє значення']))

print("\nЧас виконання процедури:")
%timeit random_sample_means(df)

Середні величини для 500 000 випадкових записів:


,Середнє значення
Sub_metering_1,1.119258
Sub_metering_2,1.308912
Sub_metering_3,6.452950



Час виконання процедури:
198 ms ± 14.7 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [5]:
def complex_filter(df):
    # 1. Час після 18:00 
    cond_time = df['Datetime'].dt.hour >= 18

    # 2. Потужність понад 6 кВт
    cond_power = df['Global_active_power'] > 6.0

    # 3. Група 2 (пралка тощо) споживає більше за Групу 1 та більше за Групу 3
    cond_group2_max = (df['Sub_metering_2'] > df['Sub_metering_1']) & (df['Sub_metering_2'] > df['Sub_metering_3'])

    # Відбираємо всі рядки, які відповідають цим трьом умовам
    filtered = df[cond_time & cond_power & cond_group2_max].reset_index(drop=True)

    # 4. Ділимо результат на дві рівні половини
    half_len = len(filtered) // 2
    first_half = filtered.iloc[:half_len]
    second_half = filtered.iloc[half_len:]

    # 5. Беремо кожен 3-й запис з першої половини і кожен 4-й з другої
    res_first = first_half.iloc[::3]
    res_second = second_half.iloc[::4]

    # Фінальна таблиця
    return pd.concat([res_first, res_second])

print("Результат складної вибірки:")
complex_df = complex_filter(df)
display(complex_df.head())
print(f"Всього відібрано записів після всіх фільтрів: {len(complex_df)}")

print("\nЧас виконання процедури:")
%timeit complex_filter(df)

Результат складної вибірки:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
0,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00
3,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00
6,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00
9,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00
12,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00


Всього відібрано записів після всіх фільтрів: 310

Час виконання процедури:
66.1 ms ± 1.84 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


Пронормувати та стандартизувати вибраний датасет


In [6]:
def normalize_and_standardize(df):
    # Лише числові колонки для математики
    num_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity']

    # Стандартизація = значення - середне / стандартне відхилення
    df_standardized = (df[num_cols] - df[num_cols].mean()) / df[num_cols].std()

    # Нормування = значення - мінімум / максимум - мінімум
    df_normalized = (df[num_cols] - df[num_cols].min()) / (df[num_cols].max() - df[num_cols].min())

    return df_standardized, df_normalized

df_stand, df_norm = normalize_and_standardize(df)

print("Стандартизовані дані (перші 5 рядків):")
display(df_stand.head())

print("\nНормовані дані (значення від 0 до 1):")
display(df_norm.head())

Стандартизовані дані (перші 5 рядків):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity
0,2.955076,2.610720,-1.851816,3.098788
1,4.037084,2.770405,-2.225274,4.133799
2,4.050325,3.320431,-2.330213,4.133799
3,4.063566,3.355916,-2.191323,4.133799
4,2.434881,3.586572,-1.592555,2.513781



Нормовані дані (значення від 0 до 1):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity
0,0.374796,0.300719,0.376090,0.377593
1,0.478363,0.313669,0.336995,0.473029
2,0.479631,0.358273,0.326010,0.473029
3,0.480898,0.361151,0.340549,0.473029
4,0.325005,0.379856,0.403231,0.323651


Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.


In [9]:
!pip install scipy

   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   - -------------------------------------- 1.6/37.3 MB 11.2 MB/s eta 0:00:04
   ---- ----------------------------------- 3.9/37.3 MB 11.4 MB/s eta 0:00:03
   ------ --------------------------------- 6.3/37.3 MB 11.6 MB/s eta 0:00:03
   --------- ------------------------------ 8.9/37.3 MB 11.7 MB/s eta 0:00:03
   ------------ --------------------------- 11.3/37.3 MB 11.7 MB/s eta 0:00:03
   -------------- ------------------------- 13.6/37.3 MB 11.7 MB/s eta 0:00:03
   ----------------- ---------------------- 16.3/37.3 MB 11.8 MB/s eta 0:00:02
   ------------------- -------------------- 18.6/37.3 MB 11.8 MB/s eta 0:00:02
   ---------------------- ----------------- 21.0/37.3 MB 11.8 MB/s eta 0:00:02
   ------------------------- -------------- 23.6/37.3 MB 11.8 MB/s eta 0:00:02
   --------------------------- ------------ 26.0/37.3 MB 11.8 MB/s eta 0:00:01
   ------------------------------ --------- 28.3/37.3 MB 11.8 MB/


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
!pip freeze > requirements.txt

In [10]:

pearson_corr = df['Global_active_power'].corr(df['Global_intensity'], method='pearson')
spearman_corr = df['Global_active_power'].corr(df['Global_intensity'], method='spearman')

print(f"Коефіцієнт Пірсона: {pearson_corr:.4f}")
print(f"Коефіцієнт Спірмена: {spearman_corr:.4f}")

Коефіцієнт Пірсона: 0.9989
Коефіцієнт Спірмена: 0.9954


Провести One Hot Encoding категоріального атрибута.

In [13]:
# Створюємо нову колонку 
df['Day_of_week'] = df['Datetime'].dt.day_name()

print("Оригінальна таблиця з новою колонкою (День тижня):")
display(df[['Datetime', 'Day_of_week']].head())

# Робимо One Hot Encoding за допомогою функції get_dummies
df_encoded = pd.get_dummies(df.head(1000), columns=['Day_of_week'])

print("\nТаблиця після One Hot Encoding (з'явилися колонки з 0 та 1 для днів):")
display(df_encoded.iloc[:, -7:].head())

Оригінальна таблиця з новою колонкою (День тижня):


,Datetime,Day_of_week
0,2006-12-16 17:24:00,Saturday
1,2006-12-16 17:25:00,Saturday
2,2006-12-16 17:26:00,Saturday
3,2006-12-16 17:27:00,Saturday
4,2006-12-16 17:28:00,Saturday



Таблиця після One Hot Encoding (з'явилися колонки з 0 та 1 для днів):


,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime,Day_of_week_Saturday,Day_of_week_Sunday
0,18.4,0.0,1.0,17.0,2006-12-16 17:24:00,True,False
1,23.0,0.0,1.0,16.0,2006-12-16 17:25:00,True,False
2,23.0,0.0,2.0,17.0,2006-12-16 17:26:00,True,False
3,23.0,0.0,1.0,17.0,2006-12-16 17:27:00,True,False
4,15.8,0.0,1.0,17.0,2006-12-16 17:28:00,True,False
